# Day 1 — The Indian Railways Network
**YTS+ DSEB 2026 · Project B**

---

Indian Railways publishes the schedule of every train in the country — which stations it stops at, in which order. Someone collected all of them: 417,080 individual station stops from 4,888 trains.

From that timetable data, you can build a map of every physical railway track in India. Today you will load the processed files, understand what they contain, and compute your first set of statistics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'

print('Ready.')

---
## Part 1 — The stations

Load the station file. Each row is one railway station.

In [ ]:
stations = pd.read_csv(DATA + 'stations_metrics.csv')

print(f'Shape: {stations.shape}')
print(f'Columns: {stations.columns.tolist()}')
print()
stations[['code', 'name', 'state', 'lat', 'lon']].head(10)

In [ ]:
by_state = stations[stations['state'] != ''].groupby('state').size().sort_values(ascending=False)

print('Stations per state (top 15):')
print(by_state.head(15).to_string())

**Q1 — Which state has the most stations?** Does that surprise you? Name one state near the bottom of the list and suggest why it has fewer stations.

*Write your answer here:*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(stations['lon'], stations['lat'], s=0.4, color='steelblue', alpha=0.5)
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'India — all {len(stations):,} railway stations')
plt.tight_layout()
plt.savefig('images/day1_stations.png', dpi=150)
plt.show()

**Q2 — Look at the map.** Where are stations densest? Where are they sparse? Name one region that looks thin and suggest a reason — geography, history, or population.

*Write your answer here:*

---
## Part 2 — The connections between stations

Load the track file. Each row is a physical track connection between two stations.

In [ ]:
track = pd.read_csv(DATA + 'track_edges_enriched.csv')

print(f'Shape: {track.shape}')
print(f'Columns: {track.columns.tolist()}')
print()
track.head(8)

In [ ]:
print('Distance between neighboring stations (km):')
print(track['distance_km'].describe().round(1))
print()
print(f'Connections under 5 km:   {(track["distance_km"] < 5).sum()}')
print(f'Connections over 100 km:  {(track["distance_km"] > 100).sum()}')
print()
print('10 longest track connections:')
longest = track.sort_values('distance_km', ascending=False)
print(longest[['name_a', 'name_b', 'distance_km']].head(10).to_string(index=False))

**Q3 — The median gap between stations is very short.** Look it up from the describe() output above. What does it tell you about who Indian Railways was built to serve — long-distance travelers or local passengers?

*Write your answer here:*

In [ ]:
print('15 busiest track connections (most trains per day):')
busiest = track.sort_values('num_trains', ascending=False)
print(busiest[['name_a', 'name_b', 'num_trains', 'distance_km']].head(15).to_string(index=False))

**Q4 — The busiest connections are all very short and clustered in one or two cities.** Which cities? What kind of trains run on these connections? What does this say about how most Indians actually use the railway?

*Write your answer here:*

---
## Part 3 — The trains

Load the train file. Each row is one train service.

In [ ]:
trains = pd.read_csv(DATA + 'train_stats.csv')

print(f'Total trains: {len(trains):,}')
print()
print('Columns:')
print('  n_stops           — how many stations this train halts at')
print('  total_distance_km — total route length in km')
print('  avg_km_per_stop   — average km between each consecutive halt')
print()
trains.head(8)

In [ ]:
print('Stops per train:')
print(trains['n_stops'].describe().round(1))
print()
print('Average km between consecutive halts:')
print(trains['avg_km_per_stop'].describe().round(1))
print()
print(f'Trains with 50+ stops:           {(trains["n_stops"] >= 50).sum()}')
print(f'Trains stopping every 5 km avg:  {(trains["avg_km_per_stop"] <= 5).sum()}')
print(f'Trains stopping every 50+ km avg:{(trains["avg_km_per_stop"] >= 50).sum()}')

In [ ]:
# Does a longer route mean fewer stops?
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(trains['total_distance_km'], trains['avg_km_per_stop'],
           s=3, alpha=0.2, color='steelblue')
ax.set_xlabel('Total route distance (km)')
ax.set_ylabel('Average km between consecutive halts')
ax.set_title('Does a longer train stop less often?')
plt.tight_layout()
plt.savefig('images/day1_express.png', dpi=150)
plt.show()

short = trains[trains['total_distance_km'] < 500]
long  = trains[trains['total_distance_km'] >= 1500]
print(f'Short trains (<500 km):   median {short["avg_km_per_stop"].median():.1f} km/stop')
print(f'Long trains  (>1500 km):  median {long["avg_km_per_stop"].median():.1f} km/stop')

**Q5 — A 2,000 km train and a 100 km train stop at almost the same frequency.** What does this tell you about how Indian Railways was designed? Who is it actually built for?

*Write your answer here:*

---
## What you did today

- Loaded three datasets: 8,697 stations, 8,612 track connections, 4,888 trains
- Computed basic distributions: station density by state, gap lengths, trains per connection
- Saw that Indian Railways is a dense, short-stop network — not primarily a long-distance express system
- Identified the busiest corridors and the longest gaps

**Next session:** build the network with networkx, compute degree and betweenness, and find the stations that control the flow.